# Searching & Sorting

Today we answer two questions that computers have to solve billions of times a day:

1. *searching*: **"Where is this thing in my list?"**
2. *sorting*: **"Can you put my list in ascending order?"**

## Warm-up: a sequence

A **sequence** is just *things in a row*, one after another. Letters in a word, students in a line,
cars in traffic. In Python we will use a **list**.

The important part: every item has a **position number**, called **index**. And **we start counting at 0.**

In [2]:
scores = [4, 8, 15, 16, 23, 42]

print("the whole list :", scores)
print("how many items :", len(scores))
print("first item     :", scores[0])
print("third item     :", scores[2])
print("last item      :", scores[len(scores) - 1])

# Walking through a list, one index at a time:
for i in range(len(scores)):
    print("index", i, "holds the value", scores[i])

the whole list : [4, 8, 15, 16, 23, 42]
how many items : 6
first item     : 4
third item     : 15
last item      : 42
index 0 holds the value 4
index 1 holds the value 8
index 2 holds the value 15
index 3 holds the value 16
index 4 holds the value 23
index 5 holds the value 42


**Everything today is built out of that loop.** A computer cannot "see" the whole list at once
the way your eyes can. It can only look at **one box at a time**.

### Run this cell first (it just makes pretty pictures of lists)

You do not need to understand this code. It draws a list with little arrows underneath so we can
watch our algorithms think.

In [1]:
def show_list(lst, marks=None, note=""):
    # marks: dictionary like {2: "^", 5: "H"} -> a label under that index
    marks = marks or {}
    widths = [max(len(str(v)), len(str(i)), 2) + 2 for i, v in enumerate(lst)]
    values = " ".join(str(v).center(w) for v, w in zip(lst, widths))
    index  = " ".join(str(i).center(w) for i, w in enumerate(widths))
    marker = " ".join(str(marks.get(i, "")).center(w) for i, w in enumerate(widths))
    print("value :", values, "  " + note)
    print("index :", index)
    if marks:
        print("        " + marker)
    print()

# quick demo
show_list([4, 8, 15, 16, 23, 42], marks={3: "^"}, note="<- looking here")

value :  4    8    15   16   23   42    <- looking here
index :  0    1    2    3    4    5  
                        ^            



---
# 1. Linear Search

## The idea

> **Start at the beginning. Check every item, one by one, until you find what you want
> (or run out of list).**

## An example, step by step

We are looking for **23** in this list:

`[4, 8, 15, 16, 23, 42]`

In [4]:
def linear_search_demo(lst, target):
    print("Searching for", target, "in", lst)
    print()
    for i in range(len(lst)):
        show_list(lst, marks={i: "^"}, note="step " + str(i + 1) + ": is " + str(lst[i]) + " == " + str(target) + " ?")
        if lst[i] == target:
            print(">>> FOUND IT at index", i, "after", i + 1, "look(s)")
            return i
    print(">>> NOT in the list (we checked all", len(lst), "boxes)")
    return -1

linear_search_demo([4, 8, 15, 16, 23, 42], 23)

Searching for 23 in [4, 8, 15, 16, 23, 42]

value :  4    8    15   16   23   42    step 1: is 4 == 23 ?
index :  0    1    2    3    4    5  
         ^                           

value :  4    8    15   16   23   42    step 2: is 8 == 23 ?
index :  0    1    2    3    4    5  
              ^                      

value :  4    8    15   16   23   42    step 3: is 15 == 23 ?
index :  0    1    2    3    4    5  
                   ^                 

value :  4    8    15   16   23   42    step 4: is 16 == 23 ?
index :  0    1    2    3    4    5  
                        ^            

value :  4    8    15   16   23   42    step 5: is 23 == 23 ?
index :  0    1    2    3    4    5  
                             ^       

>>> FOUND IT at index 4 after 5 look(s)


4

Notice the two ways the story can end:

* we find it &rarr; we **stop immediately** and report **the index**
* we fall off the end &rarr; we report **not found** or **-1**, which is our agreed code for *"not here"*

## Best case, worst case

Run the next cell and watch how much work each search costs.

In [ ]:
data = [4, 8, 15, 16, 23, 42]

linear_search_demo(data, 4)

print("=" * 60)

linear_search_demo(data, 42)

print("=" * 60)

linear_search_demo(data, 99)

Searching for 4 in [4, 8, 15, 16, 23, 42]

value :  4    8    15   16   23   42    step 1: is 4 == 4 ?
index :  0    1    2    3    4    5  
         ^                           

>>> FOUND IT at index 0 after 1 look(s)
Searching for 42 in [4, 8, 15, 16, 23, 42]

value :  4    8    15   16   23   42    step 1: is 4 == 42 ?
index :  0    1    2    3    4    5  
         ^                           

value :  4    8    15   16   23   42    step 2: is 8 == 42 ?
index :  0    1    2    3    4    5  
              ^                      

value :  4    8    15   16   23   42    step 3: is 15 == 42 ?
index :  0    1    2    3    4    5  
                   ^                 

value :  4    8    15   16   23   42    step 4: is 16 == 42 ?
index :  0    1    2    3    4    5  
                        ^            

value :  4    8    15   16   23   42    step 5: is 23 == 42 ?
index :  0    1    2    3    4    5  
                             ^       

value :  4    8    15   16   23   42    ste

-1

**Take-away:** for a list of **n** items, linear search costs **up to n looks**.

Double the list, double the work. A list of 1,000,000 names can cost 1,000,000 looks.

So... can we do better?

---
# 2. Binary Search

#### First, remember the number guessing game? What is the best strategy?

Because whatever I answer,\
you have just **thrown away half of the possible numbers**.

That instinct *is* binary search. Let's watch it play out.

## The idea

> **Look at the middle item. If it is too small, throw away the left half.
> If it is too big, throw away the right half. Repeat on what is left.**

## However, there is a limitation.





> ### Binary search only works on a **sorted** sequence.

We keep track of the part of the list still worth looking at using three markers:

* **L** = low, the left edge of what is left
* **H** = high, the right edge of what is left
* **M** = middle, the item we actually check, `M = (L + H) // 2`

## An example, step by step

Sorted list, and we are hunting for **23**:

`[4, 8, 15, 16, 23, 42, 55, 61, 78, 90]`

Predict first: how many looks will we need? (Linear search would need 5.)

In [ ]:
def binary_search_demo(lst, target):
    low, high = 0, len(lst) - 1
    step = 0
    print("Searching for", target, "in", lst)
    print()
    while low <= high:
        step += 1
        mid = (low + high) // 2
        marks = {low: "L", high: "H", mid: "M"}
        if low == mid:
            marks[low] = "LM"
        if high == mid:
            marks[high] = "MH"
        show_list(lst, marks=marks,
                  note="step " + str(step) + ": middle value is " + str(lst[mid]))
        if lst[mid] == target:
            print(">>> FOUND IT at index", mid, "after", step, "look(s)")
            return mid
        elif lst[mid] < target:
            print("    " + str(lst[mid]), "is too small -> throw away everything from index",
                  low, "to", mid)
            low = mid + 1
        else:
            print("    " + str(lst[mid]), "is too big   -> throw away everything from index",
                  mid, "to", high)
            high = mid - 1
        print()
    print(">>> NOT in the list (the search window closed up)")
    return -1

binary_search_demo([4, 8, 15, 16, 23, 42, 55, 61, 78, 90], 23)

Searching for 23 in [4, 8, 15, 16, 23, 42, 55, 61, 78, 90]

value :  4    8    15   16   23   42   55   61   78   90    step 1: middle value is 23
index :  0    1    2    3    4    5    6    7    8    9  
         L                   M                        H  

>>> FOUND IT at index 4 after 1 look(s)


4

Let's walk through this example together...

In [ ]:
# Try these too - watch the window shrink from a different direction:
binary_search_demo([4, 8, 15, 16, 23, 42, 55, 61, 78, 90], 90)
print("=" * 70)
binary_search_demo([4, 8, 15, 16, 23, 42, 55, 61, 78, 90], 50)   # not present

## What if the list is NOT sorted?

It does not crash. It does something worse: it **confidently gives a wrong answer**.

In [ ]:
messy = [42, 8, 90, 16, 4, 61, 23]      # same numbers, no order
binary_search_demo(messy, 23)           # 23 IS in there... will we find it?

Searching for 23 in [42, 8, 90, 16, 4, 61, 23]

value :  42   8    90   16   4    61   23    step 1: middle value is 16
index :  0    1    2    3    4    5    6  
         L              M              H  

    16 is too small -> throw away everything from index 0 to 3

value :  42   8    90   16   4    61   23    step 2: middle value is 61
index :  0    1    2    3    4    5    6  
                             L    M    H  

    61 is too big   -> throw away everything from index 5 to 6

value :  42   8    90   16   4    61   23    step 3: middle value is 4
index :  0    1    2    3    4    5    6  
                             MH           

    4 is too small -> throw away everything from index 4 to 4

>>> NOT in the list (the search window closed up)


-1

## Why we care: the numbers

`n` is the size of the list. Linear search costs up to `n` looks. Binary search costs about
`log2(n)` looks - the number of times you can halve `n` before you reach 1.

In [ ]:
print("list size      linear search      binary search")
print("-" * 50)
size = 10
while size <= 1000000000:
    steps = 0
    n = size
    while n > 0:
        n = n // 2
        steps += 1
    print(str(size).rjust(10), str(size).rjust(16), "looks", str(steps).rjust(8), "looks")
    size = size * 10

list size      linear search      binary search
--------------------------------------------------
        10               10 looks        4 looks
       100              100 looks        7 looks
      1000             1000 looks       10 looks
     10000            10000 looks       14 looks
    100000           100000 looks       17 looks
   1000000          1000000 looks       20 looks
  10000000         10000000 looks       24 looks
 100000000        100000000 looks       27 looks
1000000000       1000000000 looks       30 looks


Let that land: to find a name among **a billion** sorted names, binary search needs about
**30 questions**. That is the difference between a website that feels instant and one that never
loads.

**But** - it only works if the data is sorted. Sorting is the price of admission.
So how do we sort?

# Sorting

---
# 3. Bubble Sort

## The idea

> **Compare neighbours. If they are in the wrong order, swap them.
> Sweep along the list doing that, over and over, until nothing needs swapping.**

Only ever two items at a time, and they are always side by side.

Useful consequence: after sweep number `k`, the **last k items are already correct**, so the next
sweep can stop earlier.

## An example, step by step

`[5, 1, 4, 2, 8]`

Watch the 8 travel to the right. `^^` marks the pair currently being compared,
`*` marks a value that is already locked in its final position.

In [12]:
def bubble_sort_demo(lst):
    lst = list(lst)                      # work on a copy
    n = len(lst)
    print("Starting list:", lst, "\n")
    for sweep in range(n - 1):
        print("--- SWEEP", sweep + 1, "---")
        swapped = False
        for i in range(n - 1 - sweep):   # skip the locked tail
            marks = {i: "^", i + 1: "^"}
            for locked in range(n - sweep, n):
                marks[locked] = "*"
            if lst[i] > lst[i + 1]:
                show_list(lst, marks=marks,
                          note=str(lst[i]) + " > " + str(lst[i + 1]) + "  -> SWAP")
                lst[i], lst[i + 1] = lst[i + 1], lst[i]
                swapped = True
            else:
                show_list(lst, marks=marks,
                          note=str(lst[i]) + " <= " + str(lst[i + 1]) + " -> leave it")
        print("End of sweep", sweep + 1, ":", lst,
              " (index", n - 1 - sweep, "is now locked)\n")
        if not swapped:
            print("Nothing swapped in that whole sweep -> the list is already sorted. Stop early!")
            break
    print("SORTED:", lst)
    return lst

bubble_sort_demo([5, 1, 4, 2, 8])

Starting list: [5, 1, 4, 2, 8] 

--- SWEEP 1 ---
value :  5    1    4    2    8     5 > 1  -> SWAP
index :  0    1    2    3    4  
         ^    ^                 

value :  1    5    4    2    8     5 > 4  -> SWAP
index :  0    1    2    3    4  
              ^    ^            

value :  1    4    5    2    8     5 > 2  -> SWAP
index :  0    1    2    3    4  
                   ^    ^       

value :  1    4    2    5    8     5 <= 8 -> leave it
index :  0    1    2    3    4  
                        ^    ^  

End of sweep 1 : [1, 4, 2, 5, 8]  (index 4 is now locked)

--- SWEEP 2 ---
value :  1    4    2    5    8     1 <= 4 -> leave it
index :  0    1    2    3    4  
         ^    ^              *  

value :  1    4    2    5    8     4 > 2  -> SWAP
index :  0    1    2    3    4  
              ^    ^         *  

value :  1    2    4    5    8     4 <= 5 -> leave it
index :  0    1    2    3    4  
                   ^    ^    *  

End of sweep 2 : [1, 2, 4, 5, 8]  (index 3 is

[1, 2, 4, 5, 8]

### Things to point out while it runs

* The pair being compared always **touches** - index `i` and index `i+1`.
* One swap moves a value by exactly **one step**. Big values reach the end fast (they get pushed
  along by every comparison); small values crawl left one place per sweep.
* The `*` region on the right grows by one after every sweep.
* If a whole sweep makes **zero** swaps, we can quit - the list is already sorted.
  That early exit is what makes bubble sort fast on nearly-sorted data.

In [ ]:
bubble_sort_demo([1, 2, 3, 4, 5])

In [13]:
bubble_sort_demo([5, 4, 3, 2, 1])

Starting list: [5, 4, 3, 2, 1] 

--- SWEEP 1 ---
value :  5    4    3    2    1     5 > 4  -> SWAP
index :  0    1    2    3    4  
         ^    ^                 

value :  4    5    3    2    1     5 > 3  -> SWAP
index :  0    1    2    3    4  
              ^    ^            

value :  4    3    5    2    1     5 > 2  -> SWAP
index :  0    1    2    3    4  
                   ^    ^       

value :  4    3    2    5    1     5 > 1  -> SWAP
index :  0    1    2    3    4  
                        ^    ^  

End of sweep 1 : [4, 3, 2, 1, 5]  (index 4 is now locked)

--- SWEEP 2 ---
value :  4    3    2    1    5     4 > 3  -> SWAP
index :  0    1    2    3    4  
         ^    ^              *  

value :  3    4    2    1    5     4 > 2  -> SWAP
index :  0    1    2    3    4  
              ^    ^         *  

value :  3    2    4    1    5     4 > 1  -> SWAP
index :  0    1    2    3    4  
                   ^    ^    *  

End of sweep 2 : [3, 2, 1, 4, 5]  (index 3 is now locked)

[1, 2, 3, 4, 5]

---
# 4. Selection Sort

## The idea

This is the way most people sort a hand of playing cards.

> **Find the smallest item in the unsorted part. Swap it into the front of the unsorted part.
> Now that position is finished. Repeat with the rest.**

Bubble sort fixes lots of tiny local mistakes.
Selection sort makes **one big decision per round**: *who deserves this position?*

## An example, step by step

`[29, 10, 14, 37, 13]`

`P` marks the position we are filling this round, `?` marks the current
"smallest so far" candidate, and `*` marks positions already finished.

In [ ]:
def selection_sort_demo(lst):
    lst = list(lst)
    n = len(lst)
    print("Starting list:", lst, "\n")
    for pos in range(n - 1):
        print("--- ROUND", pos + 1, ": filling index", pos, "---")
        smallest = pos
        for i in range(pos + 1, n):
            marks = {j: "*" for j in range(pos)}
            marks[pos] = "P"
            marks[smallest] = "?" if smallest != pos else "P?"
            marks[i] = "^"
            show_list(lst, marks=marks,
                      note="is " + str(lst[i]) + " < " + str(lst[smallest]) + " (smallest so far)?")
            if lst[i] < lst[smallest]:
                smallest = i
                print("    yes -> new smallest is", lst[smallest], "at index", smallest)
        print("  Smallest of the unsorted part is", lst[smallest], "at index", smallest)
        if smallest != pos:
            print("  SWAP index", pos, "and index", smallest)
            lst[pos], lst[smallest] = lst[smallest], lst[pos]
        else:
            print("  Already in the right place - no swap needed")
        print("  After round", pos + 1, ":", lst, "\n")
    print("SORTED:", lst)
    return lst

selection_sort_demo([29, 10, 14, 37, 13])

Starting list: [29, 10, 14, 37, 13] 

--- ROUND 1 : filling index 0 ---
value :  29   10   14   37   13    is 10 < 29 (smallest so far)?
index :  0    1    2    3    4  
         P?   ^                 

    yes -> new smallest is 10 at index 1
value :  29   10   14   37   13    is 14 < 10 (smallest so far)?
index :  0    1    2    3    4  
         P    ?    ^            

value :  29   10   14   37   13    is 37 < 10 (smallest so far)?
index :  0    1    2    3    4  
         P    ?         ^       

value :  29   10   14   37   13    is 13 < 10 (smallest so far)?
index :  0    1    2    3    4  
         P    ?              ^  

  Smallest of the unsorted part is 10 at index 1
  SWAP index 0 and index 1
  After round 1 : [10, 29, 14, 37, 13] 

--- ROUND 2 : filling index 1 ---
value :  10   29   14   37   13    is 14 < 29 (smallest so far)?
index :  0    1    2    3    4  
         *    P?   ^            

    yes -> new smallest is 14 at index 2
value :  10   29   14   37   13    

[10, 13, 14, 29, 37]

: 

### Things to point out while it runs

* The sorted region grows from the **left**; in bubble sort it grew from the **right**.
* Each round does **at most one swap**, no matter how messy the list is. Bubble sort can do many.
* But selection sort *always* scans the entire unsorted part, even if the list is already sorted -
  so unlike bubble sort, it has **no early exit**.

In [ ]:
# Already sorted? Selection sort still does all the work:
selection_sort_demo([1, 2, 3, 4])

## Bubble vs Selection - a quick comparison

| | Bubble sort | Selection sort |
|---|---|---|
| Compares | neighbours only | one candidate against the rest |
| Swaps | many (one per fix) | at most one per round |
| Sorted region grows from | the right | the left |
| Already-sorted list | fast (early exit) | same as always |
| Comparisons | up to about n&sup2;/2 | always about n&sup2;/2 |

Both are **n&sup2;** algorithms: 10x the items means about 100x the work. Real languages use cleverer
sorts (Python's built-in `sorted()` is one), but those are built on the ideas you just learned.

---
# Practice Time
Create a new jupyter notebook (.ipynb) called search_sort.ipynb. Implement each algorithm that we covered today in a cell.
Rules of the game:

* Your code must work for **a list of any size**, including an empty list `[]`.
  Never hard-code `5` or `6`; use `len(lst)`.
* **Searches** print the **index** of the target, or **-1** if it is not there.
* **Sorts** put the list in order and **print** it.

# 1. Linear search

This could just be a warm-up for you.

### 2. Binary search

Reminder of the recipe:

1. `low = 0`, `high = len(lst) - 1`
2. While `low <= high`:
   * `mid = (low + high) // 2`
   * if `lst[mid]` **is** the target &rarr; return `mid`
   * if `lst[mid]` is **too small** &rarr; `low = mid + 1`
   * else (too big) &rarr; `high = mid - 1`
3. Fell out of the loop &rarr; return `-1`

Careful with `mid + 1` and `mid - 1`. Your program might get stuck in an
infinite loop. (If that happens: **Kernel &rarr; Interrupt**.)

### 3. Bubble sort

To swap two items in Python: `lst[i], lst[j] = lst[j], lst[i]`

### 4. Selection sort

---
## If you finish early

1. **Count the work.** Make your functions also return how many comparisons they did.
   Sort a 20-item list with both sorts and compare the counts.
2. **Sort backwards.** Change one character so the list comes out largest-first.
3. **Sort words.** Do your sorts work on `["pear", "apple", "fig"]`? Python compares strings with
   `<` just fine - try it. What order do you get, and where does `"Apple"` land?
4. **Find them all.** Write `find_all(lst, target)` that returns a *list* of every index where the
   target appears.
5. **Insertion sort.** Look it up, or invent it: take each item and slide it back into the correct
   place among the items you have already sorted. It is how most people sort a hand of cards.
6. **Race them.** Use `import time` and `time.time()` to time your bubble sort on 200, 400 and 800
   random numbers. Does 2x the data really cost about 4x the time?

---
---
# Solutions

<br><br><br><br><br><br><br><br>

In [ ]:
def linear_search_solution(lst, target):
    for i in range(len(lst)):
        if lst[i] == target:
            return i
    return -1


def binary_search_solution(lst, target):
    low, high = 0, len(lst) - 1
    while low <= high:
        mid = (low + high) // 2
        if lst[mid] == target:
            return mid
        elif lst[mid] < target:
            low = mid + 1
        else:
            high = mid - 1
    return -1


def bubble_sort_solution(lst):
    n = len(lst)
    for sweep in range(n - 1):
        swapped = False
        for i in range(n - 1 - sweep):
            if lst[i] > lst[i + 1]:
                lst[i], lst[i + 1] = lst[i + 1], lst[i]
                swapped = True
        if not swapped:
            break
    return lst


def selection_sort_solution(lst):
    n = len(lst)
    for pos in range(n - 1):
        smallest = pos
        for i in range(pos + 1, n):
            if lst[i] < lst[smallest]:
                smallest = i
        if smallest != pos:
            lst[pos], lst[smallest] = lst[smallest], lst[pos]
    return lst


print("linear    : ", end=""); check_search(linear_search_solution)
print("binary    : ", end=""); check_search(binary_search_solution, needs_sorted=True)
print("bubble    : ", end=""); check_sort(bubble_sort_solution)
print("selection : ", end=""); check_sort(selection_sort_solution)

### Common student mistakes to watch for

* **`return` inside the loop, too early.** Writing `else: return -1` in linear search reports
  "not found" after checking only the first item. The `-1` belongs *after* the loop.
* **Off-by-one at the end.** `high = len(lst)` instead of `len(lst) - 1` gives an
  `IndexError` on the very first `mid`.
* **`low = mid` instead of `mid + 1`.** Infinite loop. A good teaching moment: ask them to trace
  a 2-item list by hand.
* **Forgetting to return the list** from the sorts. The checker's message hints at this.
* **Hard-coded lengths** (`range(5)`), which is exactly what the "any size" rule is there to catch.
* **Empty list.** `len(lst) - 1` is `-1`, so `while low <= high` is `0 <= -1`, immediately false.
  It works by accident - but worth showing them *why* it works.